# Assignment 4 — Training a DCGAN on Fashion-MNIST

**Objective:** Implement and train a Deep Convolutional GAN (DCGAN) on the Fashion-MNIST
dataset, track the Generator and Discriminator losses, save 4×4 grids of generated images
every 5 epochs, and analyze how the generated images evolve over training.

**Framework:** PyTorch

**Structure of this notebook:**
1. Setup & imports
2. Load Fashion-MNIST
3. Generator (transposed convolutions)
4. Discriminator (convolutions)
5. Weight initialization
6. Training loop (30 epochs, saves losses + image grids every 5 epochs)
7. Plot Generator/Discriminator loss curves
8. Display saved image grids side-by-side to compare epochs
9. Written observations (fill in based on your own run)


## 1. Setup & Imports

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Folders to store outputs
os.makedirs("outputs/grids", exist_ok=True)
os.makedirs("outputs/checkpoints", exist_ok=True)


## 2. Load Fashion-MNIST

Images are normalized to the range **[-1, 1]** because the Generator's final activation
is `Tanh`, which also outputs values in that range.


In [ ]:
BATCH_SIZE = 128
IMAGE_SIZE = 64      # DCGAN standard, upsampled from the native 28x28
NC = 1               # number of channels (grayscale)
NZ = 100             # size of the latent (noise) vector
NGF = 64             # generator feature map base size
NDF = 64             # discriminator feature map base size
NUM_EPOCHS = 30
LR = 0.0002
BETA1 = 0.5          # Adam beta1, standard DCGAN choice

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True
)

print("Number of training images:", len(train_dataset))

# Sanity check: visualize a batch of real images
real_batch = next(iter(dataloader))
plt.figure(figsize=(6, 6))
plt.axis("off")
plt.title("Sample of real Fashion-MNIST training images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:16], padding=2,
           normalize=True, nrow=4).cpu(), (1, 2, 0)))
plt.show()


## 3. Generator (Transposed Convolutions)

The Generator takes a 100-dim latent vector `z` and upsamples it through a stack of
`ConvTranspose2d` layers into a 64×64×1 image, following the standard DCGAN architecture
(Radford et al., 2015): each block halves-the-way-in-reverse (doubles spatial size),
uses `BatchNorm2d` + `ReLU`, and the final layer uses `Tanh`.


In [ ]:
class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # input: (nz, 1, 1) -> (ngf*8, 4, 4)
            nn.ConvTranspose2d(nz, ngf * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            # (ngf*8, 4, 4) -> (ngf*4, 8, 8)
            nn.ConvTranspose2d(ngf * 8, ngf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # (ngf*4, 8, 8) -> (ngf*2, 16, 16)
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # (ngf*2, 16, 16) -> (ngf, 32, 32)
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # (ngf, 32, 32) -> (nc, 64, 64)
            nn.ConvTranspose2d(ngf, nc, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.main(z)


## 4. Discriminator (Convolutions)

The Discriminator is a standard convolutional binary classifier: it downsamples a 64×64×1
image through strided `Conv2d` layers with `BatchNorm2d` + `LeakyReLU`, ending in a single
sigmoid output (probability the image is real).


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, nc=NC, ndf=NDF):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # (nc, 64, 64) -> (ndf, 32, 32)
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf, 32, 32) -> (ndf*2, 16, 16)
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*2, 16, 16) -> (ndf*4, 8, 8)
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*4, 8, 8) -> (ndf*8, 4, 4)
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*8, 4, 4) -> (1, 1, 1)
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


## 5. Weight Initialization

Following the original DCGAN paper, all weights are initialized from a Normal
distribution (mean=0, std=0.02), which empirically stabilizes training.


In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator().to(device)
netG.apply(weights_init)

netD = Discriminator().to(device)
netD.apply(weights_init)

print(netG)
print(netD)


## 6. Loss Function & Optimizers

- Loss: Binary Cross-Entropy (`BCELoss`), the standard minimax GAN loss.
- Optimizer: Adam with `lr=0.0002`, `beta1=0.5` (the DCGAN-paper recommended settings).
- Real label = 1, Fake label = 0.
- A **fixed noise vector** is kept constant across training so we can watch how the SAME
  latent codes evolve into images epoch after epoch — this is what makes the saved grids
  comparable over time.


In [ ]:
criterion = nn.BCELoss()

fixed_noise = torch.randn(16, NZ, 1, 1, device=device)  # 16 -> 4x4 grid

REAL_LABEL = 1.0
FAKE_LABEL = 0.0

optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))


## 7. Training Loop

For every batch we:
1. **Train Discriminator**: maximize `log(D(x)) + log(1 - D(G(z)))` — one step on real images,
   one step on fake images.
2. **Train Generator**: maximize `log(D(G(z)))` (the non-saturating trick) instead of
   minimizing `log(1 - D(G(z)))`, which gives stronger gradients early in training.

Every **5 epochs** we save a 4×4 grid of images generated from the fixed noise vector, and
we record the average per-epoch loss for both networks for plotting afterward.


In [ ]:
G_losses_per_batch = []
D_losses_per_batch = []
G_losses_per_epoch = []
D_losses_per_epoch = []
saved_grid_epochs = []

def save_fixed_grid(epoch):
    netG.eval()
    with torch.no_grad():
        fake = netG(fixed_noise).detach().cpu()
    netG.train()
    grid = vutils.make_grid(fake, padding=2, normalize=True, nrow=4)
    plt.figure(figsize=(4, 4))
    plt.axis("off")
    plt.title(f"Generated images - epoch {epoch}")
    plt.imshow(np.transpose(grid, (1, 2, 0)), cmap="gray")
    path = f"outputs/grids/epoch_{epoch:03d}.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    saved_grid_epochs.append(epoch)
    return path

print("Starting Training Loop...")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0

    for i, (real_images, _) in enumerate(dataloader):
        b_size = real_images.size(0)
        real_images = real_images.to(device)

        # ---------------------
        #  Train Discriminator
        # ---------------------
        netD.zero_grad()

        # Real batch
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)
        output_real = netD(real_images)
        lossD_real = criterion(output_real, label)
        lossD_real.backward()

        # Fake batch
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake_images = netG(noise)
        label.fill_(FAKE_LABEL)
        output_fake = netD(fake_images.detach())
        lossD_fake = criterion(output_fake, label)
        lossD_fake.backward()

        lossD = lossD_real + lossD_fake
        optimizerD.step()

        # -----------------
        #  Train Generator
        # -----------------
        netG.zero_grad()
        label.fill_(REAL_LABEL)  # generator wants discriminator to say "real"
        output = netD(fake_images)
        lossG = criterion(output, label)
        lossG.backward()
        optimizerG.step()

        G_losses_per_batch.append(lossG.item())
        D_losses_per_batch.append(lossD.item())
        epoch_g_loss += lossG.item()
        epoch_d_loss += lossD.item()

    avg_g = epoch_g_loss / len(dataloader)
    avg_d = epoch_d_loss / len(dataloader)
    G_losses_per_epoch.append(avg_g)
    D_losses_per_epoch.append(avg_d)

    elapsed = time.time() - start_time
    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  Loss_D: {avg_d:.4f}  Loss_G: {avg_g:.4f}  "
          f"Elapsed: {elapsed/60:.1f} min")

    if epoch % 5 == 0 or epoch == 1:
        path = save_fixed_grid(epoch)
        print(f"  Saved 4x4 grid -> {path}")
        torch.save(netG.state_dict(), f"outputs/checkpoints/netG_epoch{epoch}.pth")
        torch.save(netD.state_dict(), f"outputs/checkpoints/netD_epoch{epoch}.pth")

print("Training finished.")


## 8. Generator & Discriminator Loss Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training (per batch)")
plt.plot(G_losses_per_batch, label="Generator", alpha=0.8)
plt.plot(D_losses_per_batch, label="Discriminator", alpha=0.8)
plt.xlabel("Training iterations (batches)")
plt.ylabel("Loss")
plt.legend()
plt.savefig("outputs/loss_curve_per_batch.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss (per-epoch average)")
plt.plot(range(1, NUM_EPOCHS + 1), G_losses_per_epoch, marker="o", label="Generator")
plt.plot(range(1, NUM_EPOCHS + 1), D_losses_per_epoch, marker="o", label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.legend()
plt.savefig("outputs/loss_curve_per_epoch.png", bbox_inches="tight")
plt.show()


## 9. Compare Generated Image Grids Across Epochs

This displays the saved 4×4 grids side by side (epoch 1, 5, 10, ... 30) so you can visually
track how the Generator's output evolves — from noise, to blurry blobs, to recognizable
clothing silhouettes, to (hopefully) sharp, diverse garment shapes.


In [ ]:
import matplotlib.image as mpimg

epochs_to_show = sorted(set(saved_grid_epochs))
n = len(epochs_to_show)
cols = min(n, 6)
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = np.array(axes).reshape(-1)

for ax, ep in zip(axes, epochs_to_show):
    img = mpimg.imread(f"outputs/grids/epoch_{ep:03d}.png")
    ax.imshow(img)
    ax.set_title(f"Epoch {ep}")
    ax.axis("off")

for ax in axes[len(epochs_to_show):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/all_epochs_comparison.png", bbox_inches="tight")
plt.show()


## 10. Written Observations (Analysis)

> Fill this section in with what you actually observe from **your own training run** —
> the notes below describe the pattern that is typical for a correctly-trained
> Fashion-MNIST DCGAN, to guide what to look for.

**Early epochs (≈1–5):**
Generated images are mostly gray/noisy blobs with no clear structure. The Discriminator
loss is usually low and the Generator loss is high at this stage, because the Discriminator
can trivially separate real clothing images from random-looking noise.

**Middle epochs (≈10–15):**
Rough silhouettes start appearing — you can usually start to tell shirts, trousers, and
shoes apart by their overall outline, even though textures and edges are still blurry or
inconsistent. This is typically when the two networks reach a more balanced back-and-forth
(neither loss dominates).

**Later epochs (≈20–30):**
Shapes become sharper and more consistent, and item categories (t-shirt, trouser, sneaker,
bag, ...) become clearly recognizable. Fine texture detail (fabric folds, logos) still
tends to stay simplified, since DCGAN is a relatively small/early architecture.

**Does quality plateau?**
Often yes — after some epoch (commonly somewhere in the 20s), the generated grids stop
visibly improving between checkpoints even though the loss values keep fluctuating. This
is expected: BCE loss values for GANs are not a reliable stand-alone quality metric, since
losses reflect the *relative* competition between G and D rather than absolute image quality.

**Mode collapse / instability check:**
Look at the 4×4 grid — if most/all 16 outputs look nearly identical (e.g., always the
same shoe silhouette), that is mode collapse: the Generator has found one output that
reliably fools the Discriminator and stopped exploring. Instability shows up as loss
curves that oscillate wildly or as the Discriminator's loss collapsing to near zero
(meaning it perfectly separates real vs fake and the Generator receives no useful gradient).
If either happens, note at which epoch it started and how the grids after that point changed
(or failed to).

**Summary takeaway:**
Training a GAN is a *competition*, not a joint minimization — losses do not need to
monotonically decrease for the model to be working. The most informative signal is the
progression of the saved image grids, not the raw loss numbers.
